# LiDAR BEV CNN 학습 · `label_2` 검증 (VoD)

**규칙 기반 파이프라인**(DBSCAN `eps`, RCS·속도 threshold, Risk 가중치 등)은 **가중치를 학습하지 않고** 사람이 값을 고르는 **튜닝**에 가깝습니다.

이 노트북은 **딥러닝 학습 파이프라인**만 다룹니다.

- **학습**: 데이터로부터 CNN 가중치를 갱신 (`BCEWithLogitsLoss` + Adam)
- **정답**: KITTI 형식 `label_2/*.txt` (캘리브로 velodyne 좌표 변환)
- **평가**: BEV 히트맵 피크 ↔ GT 중심 거리 매칭 → Precision / Recall / F1 (micro·클래스별)

구현 코드: 동일 폴더의 `bev_lidar_detector_train.py`


## 파이프라인 구조 (A ~ D)

| 단계 | 내용 |
|------|------|
| **A. 데이터셋** | LiDAR `*.bin` 로드 → BEV 격자(점 밀도·최고 z) / `label_2` 3D 중심 → 클래스별 가우시안 히트맵 타깃 |
| **B. 모델** | `TinyBevDetector`: 소형 2D CNN (BatchNorm + Conv), 출력 3채널 (Vehicle / Pedestrian / Cyclist) |
| **C. 학습 루프** | forward → loss → backward → optimizer step → epoch 반복 |
| **D. 평가** | sigmoid 히트맵에서 국소 피크 → world (x,y) vs GT greedy 매칭. **COCO 스타일 mAP**는 별도 구현이 필요함 |

### 튜닝 vs 학습 (정리)

| | 튜닝 (규칙 파이프라인) | 학습 (이 노트북) |
|--|------------------------|------------------|
| 파라미터 | DBSCAN, threshold, 가중치(고정 규칙) | **CNN 필터 가중치** |
| 데이터 사용 | 후처리·필터 | **손실 최소화로 갱신**


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# 이 노트북이 있는 폴더 = vod-devkit
NOTEBOOK_DIR = Path.cwd().resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

import bev_lidar_detector_train as bev

print("PyTorch / 모듈 준비:", NOTEBOOK_DIR)


## 데이터 루트 · 동기 프레임 인덱스

`VOD_ROOT` 환경 변수로 데이터셋 루트를 바꿀 수 있습니다. 기본은 `vod-received/view_of_delft_PUBLIC` 상대 경로입니다.


In [ ]:
DEFAULT_ROOT = NOTEBOOK_DIR / "vod-received" / "view_of_delft_PUBLIC"
DATASET_ROOT = Path(os.environ.get("VOD_ROOT", str(DEFAULT_ROOT))).resolve()

train_ratio = float(os.environ.get("BEV_SPLIT_TRAIN", "0.7"))
valid_ratio = float(os.environ.get("BEV_SPLIT_VALID", "0.15"))

frames_all = bev.list_vod_sync_frames(DATASET_ROOT)
splits = bev.split_indices(len(frames_all), train_ratio, valid_ratio)

print("dataset_root:", DATASET_ROOT)
print("exists:", DATASET_ROOT.is_dir())
print("synced frames:", len(frames_all))
print("split sizes:", {k: int(v.size) for k, v in splits.items()})
if frames_all:
    print("sample:", {k: str(v) if isinstance(v, Path) else v for k, v in frames_all[0].items()})


## 학습 · 검증 실행

| 환경 변수 | 기본 | 설명 |
|-----------|------|------|
| `BEV_MAX_TRAIN` | 400 | 학습에 쓸 최대 프레임 수 |
| `BEV_MAX_VAL` | 120 | 검증에 쓸 최대 프레임 수 |
| `BEV_EPOCHS` | 5 | 에폭 |
| `BEV_BATCH` | 4 | 배치 크기 |
| `BEV_LR` | 1e-3 | Adam 학습률 (아래 셀에서 직접 넘기려면 코드 수정) |

결과 JSON: `outputs/bev_lidar_train_metrics.json`


In [ ]:
BEV_MAX_TRAIN = int(os.environ.get("BEV_MAX_TRAIN", "400"))
BEV_MAX_VAL = int(os.environ.get("BEV_MAX_VAL", "120"))
BEV_EPOCHS = int(os.environ.get("BEV_EPOCHS", "5"))
BEV_BATCH = int(os.environ.get("BEV_BATCH", "4"))

if len(frames_all) == 0:
    raise RuntimeError("동기 프레임이 없습니다. VOD_ROOT·폴더 구조를 확인하세요.")

train_report = bev.run_train_and_eval(
    DATASET_ROOT,
    splits["train"],
    splits["valid"],
    frames_all,
    epochs=BEV_EPOCHS,
    batch_size=BEV_BATCH,
    max_train=BEV_MAX_TRAIN,
    max_val=BEV_MAX_VAL,
)

out_dir = NOTEBOOK_DIR / "outputs"
out_dir.mkdir(exist_ok=True)
out_json = out_dir / "bev_lidar_train_metrics.json"
bev.save_run_json(out_json, train_report)

print("saved:", out_json.resolve())
print(json.dumps(train_report, ensure_ascii=False, indent=2))


## (선택) BEV 격자 설정 바꾸기

`bev.BevGridConfig`로 전방 범위·격자 해상도·가우시안 시그마를 조정한 뒤, 위 학습 셀에서 `run_train_and_eval(..., bev_cfg=BevGridConfig(...))` 로 넘기면 됩니다.
